# Instructional Notebook for SHRED + Biomechanics

The following lines can be uncommented if running this notebook in Google Colab. Uncomment by highlighting lines and pressing Ctrl+/

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')
#!pip install mat73
#!git clone https://github.com/Jan-Williams/pyshred
#%cd /content/pyshred

These lines import standard packages for managing data.

In [2]:
import os
import numpy as np
import altair as alt
import pandas as pd
from processdata import TimeSeriesDataset
import models_TCN
import torch
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
import mat73
import functions as ft

# *** Update ***
Load a subject's data and manipulate the dataframe to be "tidy" = one row per time step and one column per signal. Depending on the dataset, loading and managing data will look different.

## Obtain subject's experimental data

Items to adjust before running a trial:

*   save_df - True (save) or False (don't save)
*   subject - '##'
*   activity code - AC## (this may need a different identifier depending on the dataset, just need a way to distinguish running speeds)

In [3]:
# Change subject number
subj = '02'    # 01-09
save_df = True   # True: save SHRED output dataframes only, False: don't save
trial_length = 5  # in minutes, accepts integers 1-6
frequency = 128 # in Hz, accepts integers up to 128

# adjust file path for saving if parameters are modified from 6min or 128Hz
if trial_length == 6:
    save_tag = str(frequency)+'Hz'
elif frequency == 128:
    save_tag = str(trial_length)+'min'


## Import Matlab file structure with subject's experimental data

Access directories where data is stored and will be saved. Manually set up folders before running the code block to ensure known file paths.

In [4]:
cwd = os.getcwd()
main_path = os.path.dirname(cwd) + '/Datasets' 

# Alternatively, use the below lines if using Colab
# main_path = '/content/drive/MyDrive/Colab_Notebooks/Datasets'
# main_path = cwd+'/Datasets'

dataset_path = main_path+'/Data' # sets path to dataset / raw data
dataframe_path = main_path+'/Dataframes'  # file path for saved dataframe results of test data
figure_path = main_path+'/Figures'
model_path = main_path+'/Models' # optionally, save the models that are trained
print(dataset_path)
print(dataframe_path)


/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Data
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes


Note on changing file directory and needing to save the parent directory:

https://stackoverflow.com/questions/14462833/how-can-i-go-back-to-the-previous-working-directory-after-changing-it

In [5]:
# load .mat file into pandas dataframe
load_mat = mat73.loadmat(dataset_path+'/Subject'+subj+'.mat')['Subject'+subj]
df = pd.DataFrame.from_dict(load_mat)

The example dataset contains several activities, two of which are 'Walking' and 'Running'. Access each independently.

In [6]:
#df_tmp = pd.DataFrame(data=df['Walking']['APDM_Accel']['Data'],
#                      columns = df['Walking']['APDM_Accel']['Labels'])

df_tmp = pd.DataFrame(data=df['Running']['APDM_Accel']['Data'],
                      columns = df['Running']['APDM_Accel']['Labels'])

pd.set_option('display.max_columns', None)

df_tmp.tail(5) # check that correct data was selected

Time (s) Activity Code                Waist                      \
                                  Acceleration (m/s^2)                       
                                                     x         y         z   
276631  2161.103082          23.0            -8.783041 -0.950413  4.298919   
276632  2161.110894          23.0            -8.775874 -0.959255  4.419395   
276633  2161.118706          23.0            -8.826469 -0.934233  4.339175   
276634  2161.126518          23.0            -8.849648 -0.914112  4.283415   
276635  2161.134331          23.0            -8.862103 -0.984093  4.383828   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
276631                 0.112333 -0.054528  0.003316           38.424890   
276632                 0.082718 -0.062604 -0.003309           38.302003   
276633                 0.103006 -0.051931 -0.001561           38.437986   
276634                 0.084268 -0.064363 -0.001866           38.535253   
276635                 0.078070 -0.067746 -0.008243           38.512309   

                                           Chest                      \
                            Acceleration (m/s^2)                       
               y          z                    x         y         z   
276631 -3.307839 -18.518251            -7.852923 -5.522012  2.718095   
276632 -3.306641 -18.442213            -7.875439 -5.523355  2.700218   
276633 -3.323155 -18.002071            -7.887265 -5.507988  2.697941   
276634 -3.204474 -18.329780            -7.866914 -5.492883  2.697887   
276635 -3.221997 -18.428381            -7.849070 -5.482645  2.697747   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
276631                 0.107766  0.099248  0.008457           66.820401   
276632                 0.105976  0.097592  0.003532           66.911185   
276633                 0.106083  0.099162  0.009980           66.906519   
276634                 0.110798  0.089977  0.008362           66.836417   
276635                 0.109550  0.079290  0.013229           66.586711   

                                      Left Ankle                      \
                            Acceleration (m/s^2)                       
               y          z                    x         y         z   
276631 -1.480736  19.606335            -8.839722  3.911164 -1.336907   
276632 -1.475818  19.629276            -8.839966  3.894944 -1.341463   
276633 -1.359510  19.509183            -8.846796  3.899537 -1.341462   
276634 -1.387685  19.421133            -8.835168  3.902021 -1.355148   
276635 -1.367301  19.310090            -8.828669  3.899359 -1.357429   

                                                                         \
       Angular Velocity (rad/s)                     Magnetic Field (uT)   
                              x         y         z                   x   
276631                 0.138236  0.022722  0.029028           26.963045   
276632                 0.141664  0.020117  0.024441           26.829047   
276633                 0.141689  0.017030  0.021234           26.855427   
276634                 0.137089  0.024354  0.027450           26.855002   
276635                 0.128894  0.026116  0.029017           26.823326   

                                     Right Ankle                      \
                            Acceleration (m/s^2)                       
                y         z                    x         y         z   
276631  19.800820  6.946636            -8.974209 -4.142919 -0.732089   
276632  19.825060  6.924322            -8.976873 -4.133873 -0.734219   
276633  19.698944  7.091746            -8.976549 -4.136090 -0.725430 

In [7]:
# remove units and simplify column titles
columns_str = ["_".join(df_tmp.columns[i]).replace(" ", "") for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = columns_str

l_replace = [df_tmp.columns[i].replace('(m/s^2)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(rad/s)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace

l_replace = [df_tmp.columns[i].replace('(uT)', '') for i in np.linspace(0,df_tmp.shape[1]-1,df_tmp.shape[1]).astype(int)]
df_tmp.columns = l_replace
df_tmp.head()

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,LeftFoot_Acceleration_x,LeftFoot_Acceleration_y,LeftFoot_Acceleration_z,LeftFoot_AngularVelocity_x,LeftFoot_AngularVelocity_y,LeftFoot_AngularVelocity_z,LeftFoot_MagneticField_x,LeftFoot_MagneticField_y,LeftFoot_MagneticField_z,RightFoot_Acceleration_x,RightFoot_Acceleration_y,RightFoot_Acceleration_z,RightFoot_AngularVelocity_x,RightFoot_AngularVelocity_y,RightFoot_AngularVelocity_z,RightFoot_MagneticField_x,RightFoot_MagneticField_y,RightFoot_MagneticField_z
0,0.007812,22.0,-9.776766,-0.697827,0.988534,0.090519,-0.057140,-0.005058,32.847764,24.114719,-32.463097,-9.684189,1.805644,0.650819,0.097159,0.073910,-0.015192,54.220723,-3.615137,13.377816,-9.488971,-0.915636,-2.072083,0.130793,0.017475,0.015024,33.765186,22.506846,-3.836418,-9.707371,0.348517,-1.959541,-0.017154,0.121120,0.007870,24.119824,-22.792971,20.579300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.015624,22.0,-9.779834,-0.629727,1.120072,0.096768,-0.068208,-0.006758,32.786805,23.953317,-32.587593,-9.665712,1.794331,0.653031,0.090918,0.076907,-0.010414,54.220444,-3.615654,13.327756,-9.519006,-0.856776,-2.049286,0.119683,0.020363,0.018061,33.639879,22.511589,-3.836626,-9.705521,0.338974,-1.935347,-0.010361,0.119655,0.008386,24.222964,-22.949626,20.748790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.023437,22.0,-9.750622,-0.737495,1.106857,0.087448,-0.057866,-0.011480,32.846306,24.182564,-32.476842,-9.672330,1.765049,0.657471,0.091573,0.072593,-0.006979,54.228440,-3.588266,13.364503,-9.504715,-0.917632,-2.053836,0.109135,0.020419,0.014683,33.798752,22.513139,-3.702542,-9.709912,0.366659,-1.921811,-0.009144,0.129494,0.006246,24.436704,-22.811743,20.721307,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.031249,22.0,-9.818587,-0.651224,1.041925,0.090583,-0.049229,-0.012642,32.620417,24.355620,-32.256618,-9.683271,1.760084,0.655431,0.096513,0.075121,-0.013855,54.187213,-3.729878,13.225097,-9.494960,-0.939932,-2.090318,0.113267,0.020201,0.016352,33.758904,22.517207,-3.724945,-9.729972,0.380744,-1.926151,-0.012598,0.122225,0.009280,24.483143,-22.763905,20.703241,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.039061,22.0,-9.758740,-0.682984,1.184619,0.095270,-0.058302,-0.014203,33.131889,24.436113,-32.164277,-9.683483,1.754577,0.650788,0.095920,0.072484,-0.008692,54.406781,-3.601698,13.233811,-9.493878,-0.902240,-2.074365,0.120266,0.020621,0.011659,33.813553,22.492845,-3.741917,-9.709800,0.359868,-1.926249,-0.015522,0.122770,0.004678,24.665206,-22.524455,20.825183,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Obtain data with desired activity code

In [8]:
# subject running codes [12, 13, 14] = 1.8, 2,2, 2.7 m/s
AC_path = 'AC1214'
df_1=df_tmp.loc[df_tmp['ActivityCode__']==12].dropna(axis=1,how='all')
df_2=df_tmp.loc[df_tmp['ActivityCode__']==14].dropna(axis=1,how='all')
df_3=df_tmp.loc[df_tmp['ActivityCode__']==13].dropna(axis=1,how='all')
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
138245,1080.004687,14.0,-15.149175,-2.447620,3.025381,2.959499,0.603838,-0.170548,41.415752,3.361663,-36.363086,-38.592417,12.500346,0.069753,-0.200646,-0.934100,-1.905330,56.791023,-5.443793,10.817507,-9.002607,0.473497,-2.155251,-1.737669,-0.437819,-2.224217,4.247766,41.179285,-12.834212,-18.914439,53.522359,24.681373,3.518082,2.471035,4.695741,28.505829,-20.599137,19.631191
138246,1080.012500,14.0,-19.409352,-2.577792,2.374426,3.487319,1.411864,-0.148571,41.038632,3.122850,-36.660188,-26.237922,0.766755,-3.784262,-1.069972,-1.277362,-1.335832,56.834970,-5.125031,10.593112,-6.433387,0.376558,-2.479573,-1.741116,-0.626118,-1.356809,3.355274,41.568308,-12.417896,-13.992319,18.957148,13.742815,-1.510178,1.427645,3.482053,27.305365,-21.607512,20.627904
138247,1080.020312,14.0,-30.696000,0.189094,2.359145,2.696973,1.806107,-0.229230,40.710166,3.017715,-37.376230,-15.424547,-3.397066,5.746753,-1.528512,-1.859741,0.364659,57.016644,-4.533565,10.138387,-2.349521,-0.992025,-1.160306,-1.720710,-0.846974,-0.461847,2.593408,41.686220,-11.559344,-9.432081,-6.778305,2.579193,-1.567784,0.764148,3.435861,26.752116,-22.539998,21.129850
138248,1080.028124,14.0,-41.672898,0.003049,1.428474,1.617417,1.542255,-0.895202,40.145197,3.766967,-37.410974,-15.244189,-4.019122,5.214688,-1.631013,-0.512359,0.635055,57.246497,-4.420346,9.280383,0.564467,-2.362599,0.442178,-1.746075,-0.928944,0.399219,2.284229,42.008936,-11.033846,-11.530822,-1.492381,4.215460,-0.147114,1.026828,3.942314,26.306781,-23.466704,21.039416
138249,1080.035936,14.0,-35.637824,-1.812169,5.813845,1.492804,1.752332,-1.524624,39.610010,5.367291,-35.860830,-15.933108,-10.658247,4.059921,-0.899205,0.477307,0.215329,57.123330,-4.433924,8.687706,0.602103,-1.642014,0.435292,-2.024351,-0.754144,1.037487,2.080286,42.134270,-10.563727,-12.954246,11.847749,-0.809971,1.697890,0.943375,3.924945,25.313072,-24.550582,21.016879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184322,1439.967189,14.0,2.710018,1.026106,1.167220,-0.395919,1.045166,0.088130,39.414818,-0.076855,-30.317756,-0.138083,0.915102,-3.431993,1.316504,0.299806,-1.559571,55.149514,-2.664764,18.524512,-2.894927,-46.753797,-46.479620,1.012502,1.502238,-4.352015,42.259933,9.599916,-2.302210,-18.638868,-14.011080,-0.195205,-0.263546,-0.718460,6.052163,0.638360,-42.670527,10.487453
184323,1439.975001,14.0,1.835576,1.532931,1.337739,-0.525204,1.098932,0.061403,39.240331,-0.192287,-30.434115,-0.792331,1.849059,-4.843405,1.495675,0.733669,-1.557301,55.023013,-1.986351,18.938544,-29.025741,-52.285455,-37.302166,-1.258697,2.143870,-7.259751,42.151245,10.715580,-2.083323,-18.802464,-11.863050,-0.142049,0.334889,-0.610909,5.854293,-0.693928,-42.907884,10.127602
184324,1439.982813,14.0,0.863785,1.788215,3.257821,-0.675669,0.847043,0.137868,39.372719,-0.702493,-30.221030,-1.115115,3.995017,-4.656340,1.640107,0.985129,-1.427984,54.912662,-1.312790,19.374786,-34.624132,-22.150080,2.487067,-2.361027,-0.430241,-9.748354,41.913011,12.392102,-1.27842

### Simplify dataframe

In [9]:
# trim length of trial (number of rows in df)
obs_samples_trial = trial_length*60*frequency
df_1 = df_1.tail(obs_samples_trial)
df_2 = df_2.tail(obs_samples_trial) # keep last n samples to exclude speed transitions
df_3 = df_3.tail(obs_samples_trial)

# downsample trial
obs_samples_freq = int(128/frequency)

df_1 = df_1.iloc[::obs_samples_freq,:]
df_2 = df_2.iloc[::obs_samples_freq,:]
df_3 = df_3.iloc[::obs_samples_freq,:]
df_2

,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
145927,1140.017968,14.0,0.360830,-2.699760,-1.226579,-1.869034,-1.769822,-0.539599,36.273195,12.803112,-31.947881,1.355148,-1.739694,0.551069,-1.516979,2.794643,-0.688384,55.867470,-6.944022,11.834445,-11.351534,1.278493,-7.002744,2.125843,-0.014549,-0.620747,23.885866,33.188662,-2.614791,-17.652977,13.450420,-4.826548,0.011391,-0.116158,-8.998635,25.315941,-29.720317,16.172511
145928,1140.025780,14.0,1.954548,-2.706373,-0.878774,-2.042815,-1.753388,-0.496160,36.228967,12.650706,-32.681784,2.995893,-0.141052,-0.663753,-1.896542,3.281299,-1.379039,55.667465,-7.045940,12.229578,-13.928897,-1.104565,-8.310342,2.464121,-0.127242,-1.166078,23.879078,33.059994,-3.137472,-19.161604,13.905098,-7.797603,0.750656,-0.055417,-8.765207,27.032477,-27.676267,16.095549
145929,1140.033592,14.0,3.774332,-2.751867,-0.105498,-2.032565,-1.707748,-0.404054,36.098653,12.429592,-32.970629,7.827690,-0.582017,0.870841,-2.949273,2.219581,-1.850098,55.687579,-6.875425,12.884978,-11.028181,2.922518,-7.566436,3.018787,-0.127031,-1.798897,23.786649,33.034114,-3.907350,-21.117656,15.633599,-8.686583,1.605471,-0.004525,-8.362426,28.387837,-25.696900,16.136097
145930,1140.041405,14.0,5.463308,-2.319238,0.355397,-1.919630,-1.574808,-0.263533,36.108551,11.883719,-33.451760,11.384415,-1.149696,-1.557733,-4.893807,1.260501,-1.584628,55.772306,-6.491569,13.199529,-9.616124,3.140357,-8.593081,3.194938,0.094771,-2.562110,23.583698,33.217438,-4.789559,-22.857016,19.545352,-8.120260,2.313280,0.058079,-7.882040,29.855442,-23.766511,15.996066
145931,1140.049217,14.0,6.626971,-1.983898,0.880489,-1.657953,-1.211277,-0.171715,36.102825,11.301936,-34.042233,11.256092,4.883514,-1.282328,-4.971234,0.015545,-0.504997,55.904923,-6.317635,12.965664,-9.003078,4.845436,-4.698812,3.563255,-0.038550,-3.468274,23.407273,33.440361,-5.525474,-23.951512,24.138003,-7.509693,2.717472,0.077211,-7.267202,31.096763,-21.514671,15.978182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184322,1439.967189,14.0,2.710018,1.026106,1.167220,-0.395919,1.045166,0.088130,39.414818,-0.076855,-30.317756,-0.138083,0.915102,-3.431993,1.316504,0.299806,-1.559571,55.149514,-2.664764,18.524512,-2.894927,-46.753797,-46.479620,1.012502,1.502238,-4.352015,42.259933,9.599916,-2.302210,-18.638868,-14.011080,-0.195205,-0.263546,-0.718460,6.052163,0.638360,-42.670527,10.487453
184323,1439.975001,14.0,1.835576,1.532931,1.337739,-0.525204,1.098932,0.061403,39.240331,-0.192287,-30.434115,-0.792331,1.849059,-4.843405,1.495675,0.733669,-1.557301,55.023013,-1.986351,18.938544,-29.025741,-52.285455,-37.302166,-1.258697,2.143870,-7.259751,42.151245,10.715580,-2.083323,-18.802464,-11.863050,-0.142049,0.334889,-0.610909,5.854293,-0.693928,-42.907884,10.127602
184324,1439.982813,14.0,0.863785,1.788215,3.257821,-0.675669,0.847043,0.137868,39.372719,-0.702493,-30.221030,-1.115115,3.995017,-4.656340,1.640107,0.985129,-1.427984,54.912662,-1.312790,19.374786,-34.624132,-22.150080,2.487067,-2.361027,-0.430241,-9.748354,41.913011,12.39

Only include sensor data for model training and testing; remove time and activity code columns

In [10]:
df_1_data = df_1.iloc[:,2:] 
df_2_data = df_2.iloc[:,2:] 
df_3_data = df_3.iloc[:,2:] 
df_2_data

,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z
145927,0.360830,-2.699760,-1.226579,-1.869034,-1.769822,-0.539599,36.273195,12.803112,-31.947881,1.355148,-1.739694,0.551069,-1.516979,2.794643,-0.688384,55.867470,-6.944022,11.834445,-11.351534,1.278493,-7.002744,2.125843,-0.014549,-0.620747,23.885866,33.188662,-2.614791,-17.652977,13.450420,-4.826548,0.011391,-0.116158,-8.998635,25.315941,-29.720317,16.172511
145928,1.954548,-2.706373,-0.878774,-2.042815,-1.753388,-0.496160,36.228967,12.650706,-32.681784,2.995893,-0.141052,-0.663753,-1.896542,3.281299,-1.379039,55.667465,-7.045940,12.229578,-13.928897,-1.104565,-8.310342,2.464121,-0.127242,-1.166078,23.879078,33.059994,-3.137472,-19.161604,13.905098,-7.797603,0.750656,-0.055417,-8.765207,27.032477,-27.676267,16.095549
145929,3.774332,-2.751867,-0.105498,-2.032565,-1.707748,-0.404054,36.098653,12.429592,-32.970629,7.827690,-0.582017,0.870841,-2.949273,2.219581,-1.850098,55.687579,-6.875425,12.884978,-11.028181,2.922518,-7.566436,3.018787,-0.127031,-1.798897,23.786649,33.034114,-3.907350,-21.117656,15.633599,-8.686583,1.605471,-0.004525,-8.362426,28.387837,-25.696900,16.136097
145930,5.463308,-2.319238,0.355397,-1.919630,-1.574808,-0.263533,36.108551,11.883719,-33.451760,11.384415,-1.149696,-1.557733,-4.893807,1.260501,-1.584628,55.772306,-6.491569,13.199529,-9.616124,3.140357,-8.593081,3.194938,0.094771,-2.562110,23.583698,33.217438,-4.789559,-22.857016,19.545352,-8.120260,2.313280,0.058079,-7.882040,29.855442,-23.766511,15.996066
145931,6.626971,-1.983898,0.880489,-1.657953,-1.211277,-0.171715,36.102825,11.301936,-34.042233,11.256092,4.883514,-1.282328,-4.971234,0.015545,-0.504997,55.904923,-6.317635,12.965664,-9.003078,4.845436,-4.698812,3.563255,-0.038550,-3.468274,23.407273,33.440361,-5.525474,-23.951512,24.138003,-7.509693,2.717472,0.077211,-7.267202,31.096763,-21.514671,15.978182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184322,2.710018,1.026106,1.167220,-0.395919,1.045166,0.088130,39.414818,-0.076855,-30.317756,-0.138083,0.915102,-3.431993,1.316504,0.299806,-1.559571,55.149514,-2.664764,18.524512,-2.894927,-46.753797,-46.479620,1.012502,1.502238,-4.352015,42.259933,9.599916,-2.302210,-18.638868,-14.011080,-0.195205,-0.263546,-0.718460,6.052163,0.638360,-42.670527,10.487453
184323,1.835576,1.532931,1.337739,-0.525204,1.098932,0.061403,39.240331,-0.192287,-30.434115,-0.792331,1.849059,-4.843405,1.495675,0.733669,-1.557301,55.023013,-1.986351,18.938544,-29.025741,-52.285455,-37.302166,-1.258697,2.143870,-7.259751,42.151245,10.715580,-2.083323,-18.802464,-11.863050,-0.142049,0.334889,-0.610909,5.854293,-0.693928,-42.907884,10.127602
184324,0.863785,1.788215,3.257821,-0.675669,0.847043,0.137868,39.372719,-0.702493,-30.221030,-1.115115,3.995017,-4.656340,1.640107,0.985129,-1.427984,54.912662,-1.312790,19.374786,-34.624132,-22.150080,2.487067,-2.361027,-0.430241,-9.748354,41.913011,12.392102,-1.278426,-20.085420,-9.495774,-1.082833,0.765573,-0.567552,5.491670,-2.183233,-43.181694,9.868258
184325,-0.795910,2.030456,3.632560,-0.554598,0.683752,0.212846,39

In [11]:
# convert pandas dataframe to numpy array
#load_X = df_2_data.to_numpy()
load_X = np.concatenate((df_2_data, df_1_data), axis=0)
load_XT = df_3_data.to_numpy()
load_X.shape, load_XT.shape

((76800, 36), (38400, 36))

## Set up sensors

In [12]:
from random import choice

lags = frequency # length of trajectory used to train LSTM; chose 128 for Ingraham data sampled at 128 Hz
n = load_X.shape[0] # total number of time steps (observations)
m = load_X.shape[1] # number of features per time step

time = np.arange(1, n+1, 1)

## Visualize IMU data

Observing raw data is important for understanding what is being used to train and test models. We visualize data using altair (alt). Two tutorials on some basic functionality are linked below:

* Long tutorial (1hr): https://youtu.be/umTwkgQoo_E

* Short tutorial (20min): https://youtu.be/o-nVM_FdIVc

Uncomment the line below when code is fully functioning to disable the 5000-row dataframe limit

In [13]:
# alt.data_transformers.disable_max_rows()

In [14]:
# set time to start at 0 (optional for clean viz)
time_zeroed = df_2.loc[:,"Time(s)__"] - df_2["Time(s)__"].iloc[0]

# view the first portion of the trial
df_2_data_reduced = df_2.head(1000)
df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)

# view the last portion of the trial
#df_2_data_reduced = df_2.tail(4000) 
#df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.tail(4000)

df_2_data_reduced.head()

/tmp/ipykernel_2397967/3481633552.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2_data_reduced['Time_Zeroed(s)'] = time_zeroed.head(1000)


,Time(s)__,ActivityCode__,Waist_Acceleration_x,Waist_Acceleration_y,Waist_Acceleration_z,Waist_AngularVelocity_x,Waist_AngularVelocity_y,Waist_AngularVelocity_z,Waist_MagneticField_x,Waist_MagneticField_y,Waist_MagneticField_z,Chest_Acceleration_x,Chest_Acceleration_y,Chest_Acceleration_z,Chest_AngularVelocity_x,Chest_AngularVelocity_y,Chest_AngularVelocity_z,Chest_MagneticField_x,Chest_MagneticField_y,Chest_MagneticField_z,LeftAnkle_Acceleration_x,LeftAnkle_Acceleration_y,LeftAnkle_Acceleration_z,LeftAnkle_AngularVelocity_x,LeftAnkle_AngularVelocity_y,LeftAnkle_AngularVelocity_z,LeftAnkle_MagneticField_x,LeftAnkle_MagneticField_y,LeftAnkle_MagneticField_z,RightAnkle_Acceleration_x,RightAnkle_Acceleration_y,RightAnkle_Acceleration_z,RightAnkle_AngularVelocity_x,RightAnkle_AngularVelocity_y,RightAnkle_AngularVelocity_z,RightAnkle_MagneticField_x,RightAnkle_MagneticField_y,RightAnkle_MagneticField_z,Time_Zeroed(s)
145927,1140.017968,14.0,0.360830,-2.699760,-1.226579,-1.869034,-1.769822,-0.539599,36.273195,12.803112,-31.947881,1.355148,-1.739694,0.551069,-1.516979,2.794643,-0.688384,55.867470,-6.944022,11.834445,-11.351534,1.278493,-7.002744,2.125843,-0.014549,-0.620747,23.885866,33.188662,-2.614791,-17.652977,13.450420,-4.826548,0.011391,-0.116158,-8.998635,25.315941,-29.720317,16.172511,0.000000
145928,1140.025780,14.0,1.954548,-2.706373,-0.878774,-2.042815,-1.753388,-0.496160,36.228967,12.650706,-32.681784,2.995893,-0.141052,-0.663753,-1.896542,3.281299,-1.379039,55.667465,-7.045940,12.229578,-13.928897,-1.104565,-8.310342,2.464121,-0.127242,-1.166078,23.879078,33.059994,-3.137472,-19.161604,13.905098,-7.797603,0.750656,-0.055417,-8.765207,27.032477,-27.676267,16.095549,0.007812
145929,1140.033592,14.0,3.774332,-2.751867,-0.105498,-2.032565,-1.707748,-0.404054,36.098653,12.429592,-32.970629,7.827690,-0.582017,0.870841,-2.949273,2.219581,-1.850098,55.687579,-6.875425,12.884978,-11.028181,2.922518,-7.566436,3.018787,-0.127031,-1.798897,23.786649,33.034114,-3.907350,-21.117656,15.633599,-8.686583,1.605471,-0.004525,-8.362426,28.387837,-25.696900,16.136097,0.015624
145930,1140.041405,14.0,5.463308,-2.319238,0.355397,-1.919630,-1.574808,-0.263533,36.108551,11.883719,-33.451760,11.384415,-1.149696,-1.557733,-4.893807,1.260501,-1.584628,55.772306,-6.491569,13.199529,-9.616124,3.140357,-8.593081,3.194938,0.094771,-2.562110,23.583698,33.217438,-4.789559,-22.857016,19.545352,-8.120260,2.313280,0.058079,-7.882040,29.855442,-23.766511,15.996066,0.023437
145931,1140.049217,14.0,6.626971,-1.983898,0.880489,-1.657953,-1.211277,-0.171715,36.102825,11.301936,-34.042233,11.256092,4.883514,-1.282328,-4.971234,0.015545,-0.504997,55.904923,-6.317635,12.965664,-9.003078,4.845436,-4.698812,3.563255,-0.038550,-3.468274,23.407273,33.440361,-5.525474,-23.951512,24.138003,-7.509693,2.717472,0.077211,-7.267202,31.096763,-21.514671,15.978182,0.031249


Select which sensor location to visualize.

In [15]:
location = 'RightAnkle' # RightAnkle, LeftAnkle, Chest, Waist

In [16]:
# Define signal types and axes.
sensor = ['Acceleration', 'AngularVelocity', 'MagneticField']
dir = ['x','y','z']
plotStack = [0,0,0] # Preallocate plot for each signal

# Generate plots for each sensor type
for iSensor in range(len(sensor)): # loop through the signal types
    # create plots for x,y,z directions
    x_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_x', title = sensor[iSensor]),
        color = alt.value('#c6dbef')
    ).properties(
        width = 1000,
        height = 200
    )
    y_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_y', title = sensor[iSensor]),
        color = alt.value("#6baed6")
    )
    z_signal = alt.Chart(df_2_data_reduced).mark_line().encode(
        x = 'Time_Zeroed(s)',
        y = alt.Y(location + '_' + sensor[iSensor] + '_z', title = sensor[iSensor]),
        color = alt.value("#08519c")
    ).interactive()
    # Combine x,y,z plots
    plotStack[iSensor] = x_signal + y_signal + z_signal

alt.vconcat(plotStack[0], plotStack[1], plotStack[2]).properties(title = [location,""])

alt.VConcatChart(...)

# SHRED model function

In [17]:
### Generate input sequences to a SHRED model
def train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags):
  """
    Trains SHRED model for time series reconstruction

  Args: 
    transformed_X (numpy array): MinMax scaled dataset.
    sc (MinMaxScaler): Fitted MinMax scaler for inverse transformation.
    train_indices (array): Indices for the training set.
    valid_indices (array): Indices for the validation set.
    test_indices (array): Indices for the test set.
    sensor_locations (array): Column indices for sensor data.
    num_sensors (int): Number of signal measurements from sensors (e.g, triaxial = 3)
    m (int): Number of features per timestep
    n (int): Total number of time steps (observations)
    lags (int): length of trajectory
    
  Return:
    test_recons: Reconstructed data from the SHRED model on the test set
    test_ground_truth: Ground truth data from the test set
  """

  all_data_in = np.zeros((n - lags, lags, num_sensors))
  for i in range(len(all_data_in)):
      all_data_in[i] = transformed_X[i:i+lags, sensor_locations]
  ### Generate training validation and test datasets both for reconstruction of states and forecasting sensors
  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
  valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
  test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

  ### -1 to have output be at the same time as final sensor measurements
  train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
  valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
  test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

  train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
  valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
  test_dataset = TimeSeriesDataset(test_data_in, test_data_out)


  ##Modify this part for TCN
  #shred = models_TCN.SHRED_TCN(num_sensors, m, number_channels=2, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
  tcn_chaneels = [64, 64]
  shred = models_TCN.SHRED_TCN(num_sensors, m, num_channels=tcn_chaneels, fc_layers=[350, 400], kernel_size=3, dropout=0.2).to(device)

  validation_errors = models_TCN.fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=500, lr=1e-3, verbose=True, patience=3)

  # Generate reconstructions from the test set and print mean square error compared to the ground truth
  test_recons = sc.inverse_transform(shred(test_dataset.X).detach().cpu().numpy())
  test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())

  return test_recons, test_ground_truth

# Train models

Divide the data into training, validation and test.

In [18]:
# partition into training, validation, test sets
#train_indices, valid_indices, test_indices = ft.partition_data_seq(load_X, n, lags) 
train_indices, valid_indices, _ = ft.partition_data_seq(load_X, n, lags) 
test_data = load_XT


# normalize input data using MinMaxScaler
transformed_X, sc = ft.transform_data(load_X, train_indices) 
transformed_XT = sc.transform(test_data)
test_indices = np.arange(transformed_XT.shape[0])

### Define input sensor

In [19]:
# choose input sensor location
sensor_place = 'RightAnkle' # RightAnkle, Waist, or Chest

# choose input sensor type
sensor_path = '3acc_Training' # 3acc_Training, 3gyro_Training, 3acc3gyro_Training, or Xacc_Training

# access columns indices from main dataframe
sensor_locations, num_sensors = ft.sensor_loc_fun(sensor_path, sensor_place, df_2_data) # This function is specific to the dataset used in this project. Update it according the the types of signals (joint angles, EMG, etc) in your dataset.
train_names = [df_2_data.columns[i] for i in sensor_locations]

print('Number of signals: ', num_sensors)
print('Signals were chosen at: ', sensor_place)
print('Signals chosen: ', [df_2_data.columns[i] for i in sensor_locations])

Number of signals:  3
Signals were chosen at:  RightAnkle
Signals chosen:  ['RightAnkle_Acceleration_x', 'RightAnkle_Acceleration_y', 'RightAnkle_Acceleration_z']


In [20]:
# check path for saving dataframes
if trial_length == 6 and frequency == 128: # full-length trial, full frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ytest_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'.csv'
else: # reduced trial length or frequency
  save_train_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'
  save_test_df = dataframe_path+'/'+AC_path+'/'+sensor_place+'/'+sensor_path+'/P'+subj+'_Ypred_SHRED_Train_'+sensor_place+'_'+sensor_path+'_'+save_tag+'.csv'

print(os.path.isdir(save_test_df))
print(save_train_df)
print(save_test_df)

False
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC1214/RightAnkle/3acc_Training/P02_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv
/mnt/ssd1/wyc/my_projects/SHRED/SHRED/Datasets/Dataframes/AC1214/RightAnkle/3acc_Training/P02_Ypred_SHRED_Train_RightAnkle_3acc_Training_5min.csv


In [21]:
test_data_length = len(load_XT)
test_time_indices = np.arange(0, test_data_length)
test_times = df_3_data.iloc[test_time_indices,0].to_numpy()

In [22]:
test_times.shape, test_time_indices.shape, load_XT.shape

((38400,), (38400,), (38400, 36))

In [23]:
# train SHRED model
Ypred, Ytest = train_SHRED_model(transformed_X, sc, train_indices, valid_indices, test_indices, sensor_locations, num_sensors, m, n, lags)

df_Ytest_SHRED = pd.DataFrame(Ytest, columns = df_2_data.columns)
df_Ypred_SHRED = pd.DataFrame(Ypred, columns = df_2_data.columns)

df_Ytest_SHRED['Type']='Measured'
df_Ypred_SHRED['Type']='TCN'

df_Ytest_SHRED['Time']=  test_times          # df_2.iloc[test_indices + lags - 1,0].to_numpy()
df_Ypred_SHRED['Time']=  test_times         # df_2.iloc[test_indices + lags - 1,0].to_numpy()

# save dataframes as .csv if specified
if save_df == True:
  df_Ytest_SHRED.to_csv(save_train_df)
  df_Ypred_SHRED.to_csv(save_test_df)


/mnt/ssd1/wyc/SHREDwyc/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Training epoch 1
Error tensor(0.1572, device='cuda:0')
Training epoch 20
Error tensor(0.0851, device='cuda:0')
Training epoch 40
Error tensor(0.0788, device='cuda:0')
Training epoch 60
Error tensor(0.0783, device='cuda:0')
Training epoch 80
Error tensor(0.0795, device='cuda:0')
Training epoch 100
Error tensor(0.0770, device='cuda:0')
Training epoch 120
Error tensor(0.0777, device='cuda:0')
Training epoch 140
Error tensor(0.0772, device='cuda:0')
Training epoch 160
Error tensor(0.0787, device='cuda:0')


# Visualize Results

In [24]:
df_SHRED_tidy = ft.concatRaw(1,sensor_place, sensor_path, 1, 1, df_Ypred_SHRED, df_Ytest_SHRED ,subj)

In [25]:
print("df_SHRED_tidy 的列名:", df_SHRED_tidy.columns)
print("df_SHRED_tidy 的头部数据:\n", df_SHRED_tidy.head())

df_SHRED_tidy 的列名: Index(['Subject', 'Pred', 'True', 'Input Location', 'Sensor Type',
       'Output Location', 'Output Signal', 'Output Direction', 'Output Axis',
       'Assessment', 'Time'],
      dtype='object')
df_SHRED_tidy 的头部数据:
   Subject       Pred       True Input Location             Sensor Type  \
0      02 -28.898134 -29.335743     RightAnkle  Triaxial Accelerometer   
1      02  -3.337169  -3.944044     RightAnkle  Triaxial Accelerometer   
2      02   3.590603   0.092497     RightAnkle  Triaxial Accelerometer   
3      02   1.907615   2.047456     RightAnkle  Triaxial Accelerometer   
4      02   0.509104   0.677225     RightAnkle  Triaxial Accelerometer   

  Output Location     Output Signal Output Direction Output Axis  \
0           Waist      Acceleration         Vertical           x   
1           Waist      Acceleration               AP           y   
2           Waist      Acceleration               ML           z   
3           Waist  Angular Velocity         V

In [26]:
# format: ft.extractSignal(output_location, output_signal, output_axis, df_SHRED_tidy)
    # output_location: 'Chest', 'Waist', 'RightAnkle', 'LeftAnkle'
    # output_signal: 'Acceleration', 'Angular Velocity', 'Magnetic Field'
    # output_axis: 'x', 'y', z'

Signal1 = ft.extractSignal('LeftAnkle', 'Angular Velocity', 'x', df_SHRED_tidy)
Signal2 = ft.extractSignal('Chest', 'Angular Velocity', 'x', df_SHRED_tidy)
Signal3 = ft.extractSignal('Waist', 'Angular Velocity', 'x', df_SHRED_tidy)

my_scheme = ['#1e88e5', "#6E6E6E"] # '#014337', '#1e88e5', '#DB1048'

# Compute error: ft.rmse_error, ft.mae_error, OR ft.mbe_error
Signal1_error = ft.rmse_error(Signal1[Signal1['Type'] == 'True']['Value'], Signal1[Signal1['Type'] == 'SHRED']['Value'])
Signal2_error = ft.rmse_error(Signal2[Signal2['Type'] == 'True']['Value'], Signal2[Signal2['Type'] == 'SHRED']['Value'])
Signal3_error = ft.rmse_error(Signal3[Signal3['Type'] == 'True']['Value'], Signal3[Signal3['Type'] == 'SHRED']['Value'])

# plot left ankle acceleration
line1 = alt.Chart(Signal1).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 1 RightAnkle: RMSE = {Signal1_error:.2f}'  # Can change this title to be specific to the output signal
)
# plot chest acceleration
line2 = alt.Chart(Signal2).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 2: Chest RMSE = {Signal2_error:.2f}'  # Can change this title to be specific to the output signal
)

# plot Waist acceleration
line3 = alt.Chart(Signal3).mark_line().encode(
    x=alt.X('Time:Q', title='Time (s)'),
    y=alt.Y('Value:Q', title='Acceleration (m/s\u00b2)'),
    color=alt.Color('Type:N', scale=alt.Scale(range=my_scheme))
).properties(
    width=600,
    height=400,
    title=f'Signal 3: Waist RMSE = {Signal3_error:.2f}'  # Can change this title to be specific to the output signal
)

final_chart = alt.vconcat(line3, line2, line1).properties(
    title=f'Parameter = {save_tag}, Right Ankle Input' # Can change this title to match the input sensor
    # increase font size
).configure_axis(
    labelFontSize=18,
    titleFontSize=20
).configure_title(
    fontSize=24
).configure_legend(
    labelFontSize=18,
    titleFontSize=20
)

final_chart

alt.VConcatChart(...)